# 🎯 Word-Level Training — CPU (Simple & Reliable)
**Using:** Pre-installed PyTorch on Kaggle CPU
**Why:** CUDA compatibility issues on Kaggle P100 with GPU mode
**Time:** ~5 min per fold, ~25 min total

**Data:** 48 EMNLP videos, 39,944 words, 10.9% positive rate


In [ ]:
# Cell 1: Setup
import os, warnings, time
warnings.filterwarnings('ignore')

DATA_BASE = '/kaggle/input/datasets/subhajitdas/chucklenet-48v-wordlevel'
FEAT_DIR = f'{DATA_BASE}/features'
LABEL_DIR = f'{DATA_BASE}/labels'

print(f'Features exist: {os.path.exists(FEAT_DIR)}')
print(f'Labels exist: {os.path.exists(LABEL_DIR)}')

import numpy as np
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# Cell 2: Load data
X_list, y_list, vids_list = [], [], []

for f in sorted(os.listdir(FEAT_DIR)):
    if not f.endswith('_features.npy'): continue
    vid = f.replace('_features.npy', '')
    label_path = f'{LABEL_DIR}/{vid}_labels.npy'
    if not os.path.exists(label_path): continue
    try:
        X = np.load(f'{FEAT_DIR}/{f}')
        y = np.load(label_path)
        n = min(len(X), len(y))
        X_list.append(X[:n])
        y_list.append(y[:n])
        vids_list.extend([vid] * n)
    except Exception as e:
        print(f'Error {vid}: {e}')
        continue

X = np.vstack(X_list).astype(np.float32)
y = np.concatenate(y_list)
vids = np.array(vids_list)
X = np.nan_to_num(X, nan=0.0)

unique_vids = sorted(set(vids_list))
pos_rate = y.mean()

print(f'Videos: {len(unique_vids)}, Words: {len(y)}')
print(f'Positive rate: {100*pos_rate:.1f}%')

In [ ]:
# Cell 3: Train
class MLP(nn.Module):
    def __init__(self, d=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

t0 = time.time()
gkf = GroupKFold(n_splits=min(5, len(unique_vids)))
pw = min((1-pos_rate)/max(pos_rate, 1e-6), 3.0)
print(f'pos_weight: {pw:.2f}')

fold_f1s = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, vids)):
    Xtr, Xte = X[tr_idx], X[te_idx]
    ytr, yte = y[tr_idx], y[te_idx]
    if yte.sum()==0 or ytr.sum()==0: continue

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr).astype(np.float32)
    Xte_s = scaler.transform(Xte).astype(np.float32)

    model = MLP(d=X.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

    Xtr_t = torch.tensor(Xtr_s)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)

    best_f1, patience, no_imp = 0, 10, 0
    for ep in range(60):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(Xtr_t), 128):
            idx = perm[i:i+128]
            if len(idx) < 2: continue
            opt.zero_grad()
            out = model(Xtr_t[idx])
            w = torch.where(ytr_t[idx]==1, pw, 1.0)
            loss = -(w * (ytr_t[idx]*torch.log(out+1e-8) + (1-ytr_t[idx])*torch.log(1-out+1e-8))).mean()
            loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            probs = model(torch.tensor(Xte_s)).squeeze().numpy()
        f = f1_score(yte, (probs>=0.5).astype(int), zero_division=0)
        if f > best_f1: best_f1 = f; no_imp = 0
        else: no_imp += 1
        if no_imp >= patience: break

    model.eval()
    with torch.no_grad():
        probs = model(torch.tensor(Xte_s)).squeeze().numpy()
    p = precision_score(yte, (probs>=0.5).astype(int), zero_division=0)
    r = recall_score(yte, (probs>=0.5).astype(int), zero_division=0)
    f = f1_score(yte, (probs>=0.5).astype(int), zero_division=0)
    fold_f1s.append(f)
    print(f'Fold {fold+1}: F1={f:.4f} P={p:.4f} R={r:.4f} ({time.time()-t0:.0f}s)')

print(f'\nCV F1: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}')
print(f'Total time: {time.time()-t0:.0f}s')

In [ ]:
# Cell 4: Save
import json
torch.save(model.state_dict(), '/kaggle/working/word_level_model.pt')
results = {
    'n_videos': len(unique_vids),
    'n_words': int(len(y)),
    'positive_rate': float(pos_rate),
    'cv_f1': float(np.mean(fold_f1s)),
    'cv_std': float(np.std(fold_f1s)),
    'fold_f1s': [float(f) for f in fold_f1s]
}
with open('/kaggle/working/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))